# Boltz-2 Structure Prediction

This notebook demonstrates multi-modal structure prediction using Boltz-2, an open-source model developed by MIT. Boltz-2 predicts the three-dimensional structures of proteins, nucleic acids (DNA and RNA), small-molecule ligands, and their complexes using a diffusion-based generative approach with configurable recycling and sampling steps. We demonstrate `run_boltz2` by predicting the structure of the MfnG protein bound to L-tyrosine, covering input construction, model configuration, result inspection, visualization, and export.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("boltz2")
display_overview("boltz2")
display_docs_section("boltz2", "Background")

# Boltz-2

Boltz-2 is an openly licensed biomolecular structure prediction model from the [MIT Jameel Clinic](https://jclinic.mit.edu/) and [Recursion](https://www.recursion.com/), built in the AlphaFold3 family: a diffusion model that predicts the joint 3D structure of complexes mixing proteins, DNA, RNA, and small-molecule ligands, together with the binding affinity of a small molecule against a protein target. This toolkit runs Boltz-2 structure and affinity prediction on a local GPU, with optional ColabFold multiple-sequence alignments.

Boltz-2 ([Passaro et al., 2025](https://doi.org/10.1101/2025.06.14.659707)) predicts the joint 3D structure of a biomolecular assembly from the sequences and chemical components it contains. It builds on Boltz-1, one of the most widely used open-source alternatives to AlphaFold3, extending that co-folding model with a binding-affinity module, improved controllability, and additional training data. Like AlphaFold3, a single model folds complexes that mix proteins, DNA, RNA, and small-molecule ligands and predicts how those components are arranged relative to one another. Each protein chain can be paired with a multiple-sequence alignment (MSA) of evolutionarily related sequences, whose covariation patterns supply the evolutionary signal the model uses to place residues.

Architecturally, Boltz-2 reproduces AlphaFold3: it carries a single representation of the input tokens and a pairwise representation over token pairs, refines them through an AlphaFold3-style trunk, and generates all-atom coordinates with a diffusion module that starts from noise and iteratively denoises into a structure. Several structures can be sampled per complex and ranked by a confidence score, reported as a complex predicted local distance difference test (pLDDT) for local reliability, a predicted aligned error (PAE) for the relative placement of any two tokens, and predicted template-modeling (pTM) and interface predicted template-modeling (ipTM) scores that summarize overall and interface accuracy. Beyond structure, Boltz-2 adds a binding-affinity module that approaches the accuracy of physics-based free-energy perturbation while running more than 1000 times faster.

The reference implementation is open-sourced at [jwohlwend/boltz](https://github.com/jwohlwend/boltz) under the MIT license, covering the code, weights, and training pipeline for both academic and commercial use, with the released weights distributed as `boltz-community/boltz-2`. It was developed by the Boltz team at the [MIT Jameel Clinic](https://jclinic.mit.edu/) together with [Recursion](https://www.recursion.com/).

## Available tools

In [2]:
display_available_tools("boltz2")

- **`run_boltz2_affinity()`** — Predicted binding affinity (log10 IC50 μM) and binder probability for a small molecule against a protein target, via Boltz-2.
- **`run_boltz2()`** — Multi-modal structure prediction using Boltz2

### `run_boltz2`

Boltz-2 predicts 3D structures of proteins, DNA, RNA, ligands, and their complexes using a diffusion-based generative model. It supports both single-sequence mode and MSA-assisted prediction via ColabFold search. The `recycling_steps` parameter controls iterative structural refinement, while `sampling_steps` governs the granularity of the denoising process. When `diffusion_samples` is set above 1, Boltz-2 generates multiple independent structure samples and returns the best by confidence score, which is particularly useful for exploring conformational diversity in flexible protein-ligand systems. Ligands are provided as SMILES strings and are automatically converted to the appropriate internal representation.

In [3]:
from pathlib import Path

from proto_tools import (
    Boltz2Config,
    Boltz2Input,
    Chain,
    Complex,
    run_boltz2,
)

In [4]:
# Display input docs
display_api_reference("boltz2", "input", "run_boltz2")

# MfnG protein sequence (enzyme from methanogenic archaea)
mfng_sequence = "MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDALYAYDVIDRIHDTNGFRYLLSAEARECLLPGTLFSLVGKFMHDINVAWPAWRNLAEVVRHGARDTSGAESPNGIAQEDYESLVGGINFWAPPIVTTLSRKLRASGRSGDATASVLDVGCGTGLYSQLLLREFPRWTATGLDVERIATLANAQALRLGVEERFATRAGDFWRGGWGTGYDLVLFANIFHLQTPASAVRLMRHAAACLAPDGLVAVVDQIVDADREPKTPQDRFALLFAASMTNTGGGDAYTFQEYEEWFTAAGLQRIETLDTPMHRILLARRATEPSAVPEGQASENLYFQ"

# L-tyrosine SMILES
tyrosine_smiles = "c1cc(ccc1C[C@@H](C(=O)O)N)O"

# Create protein-ligand complex
complex = Complex(
    chains=[
        Chain(sequence=mfng_sequence, entity_type="protein"),
        Chain(sequence=tyrosine_smiles, entity_type="ligand"),
    ]
)

# Create input
inputs = Boltz2Input(complexes=[complex])

**Input** — `Boltz2Input`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `complexes` | `list[proto_tools.entities.complex.Complex]` | required | List of complexes to predict structure for containing chains and entity types. |
| `msas` | `list[proto_tools.tools.structure_prediction.shared_data_models.ComplexMSAs] \| None` | `None` | Per-complex MSAs; a bare dict[int, MSA] is coerced to an unpaired ComplexMSAs. |

In [5]:
# Display config docs
display_api_reference("boltz2", "config", "run_boltz2")

# Configure Boltz-2 with reduced settings for a fast demonstration run
config = Boltz2Config(
    verbose=False,
    device="cuda",  # Change to "cpu" if no GPU available
    use_msa=True,
    recycling_steps=3,
    diffusion_samples=1,
)

**Config** — `Boltz2Config`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on (e.g., 'cuda', 'cpu') |
| `timeout` | `int \| None` | `1200` | Maximum execution time in seconds. |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `include_pae_matrix` | `bool` | `False` | Attach the full per-residue PAE matrix. |
| `use_msa` | `bool` | `True` | Whether to generate and use MSAs for protein chains using ColabFold search |
| `colabfold_search_config` | `proto_tools.tools.sequence_alignment.colabfold_search.colabfold_search.ColabfoldSearchConfig \| None` | `None` | Nested configuration for ColabFold MSA search. If None, uses default settings. |
| `recycling_steps` | `int` | `3` | Iterative refinement passes through the model. Higher = more accurate but slower. |
| `sampling_steps` | `int` | `200` | Denoising steps in the diffusion process. Higher = more refined but slower. |
| `diffusion_samples` | `int` | `1` | Structure samples per complex; best by confidence is kept. Higher = more thorough but slower. |
| `step_scale` | `float` | `1.5` | Diffusion step size (typical range 1.0-2.0). Lower values produce more sample diversity. |
| `max_msa_seqs` | `int` | `8192` | Maximum number of MSA sequences fed into the model. Lower this to reduce GPU memory on deep MSAs. |
| `subsample_msa` | `bool` | `False` | Randomly subsample the MSA on each run for sample diversity (loses determinism). |
| `num_workers` | `int` | `4` | Number of dataloader workers for prediction. |

In [6]:
# Run structure prediction
result = run_boltz2(inputs, config)

Running run_colabfold_search [00:00]

Folding structures (Boltz-2):   0%|          | 0/1 [00:00<?, ?complex/s]

In [7]:
# Display output docs
display_api_reference("boltz2", "output", "run_boltz2")

mfng_structure = result.structures[0]

# Print confidence metrics
print(f"  Number of chains:  {len(complex.chains)}")
print(f"  Protein length:    {len(mfng_sequence)} residues")
print(f"  Confidence score:  {mfng_structure.metrics.confidence_score:.3f}")
print(f"  Complex pLDDT:     {mfng_structure.metrics.complex_plddt:.3f}")
print(f"  pTM score:         {mfng_structure.metrics.ptm:.3f}")
print(f"  ipTM score:        {mfng_structure.metrics.iptm:.3f}")
print(f"  Ligand ipTM:       {mfng_structure.metrics.ligand_iptm:.3f}")

**Output** — `Boltz2Output`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `structures` | `list[proto_tools.entities.structures.structure.Structure]` | required | List of predicted structures, one per input complex. |

  Number of chains:  2
  Protein length:    384 residues
  Confidence score:  0.913
  Complex pLDDT:     0.929
  pTM score:         0.919
  ipTM score:        0.848
  Ligand ipTM:       0.848


#### Visualize the predicted complex

The interactive viewer renders the predicted protein-ligand complex colored by chain, allowing you to inspect the binding pose and overall fold geometry.

> NOTE: The 3D viewer below renders locally (JupyterLab, VS Code) but not in GitHub previews.

In [8]:
mfng_structure.visualize(style="cartoon", color_by="bfactor")

## Export Results

Predicted structures can be exported to PDB or mmCIF format for downstream analysis in molecular visualization tools such as PyMOL, ChimeraX, or VMD. The B-factor column contains pLDDT confidence scores for per-residue visualization.

In [9]:
# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export results to PDB files
result.export(name="mfng_ligand_complex", export_path=output_dir, file_format="pdb")
print(f"Structure exported to {output_dir / 'mfng_ligand_complex.pdb'}")

Structure exported to example_output/mfng_ligand_complex.pdb


## Predict Binding Affinity

Boltz-2's affinity head predicts the binding affinity of a single small-molecule ligand against a protein target. `run_boltz2_affinity` runs the same diffusion model with an added `properties.affinity.binder` block, so the predicted binding pose is returned alongside the affinity scores. The binder is auto-detected here because the complex has a single ligand; set `binder_chain` to name it when a complex has several.

In [10]:
from proto_tools import Boltz2AffinityConfig, Boltz2AffinityInput, run_boltz2_affinity

display_api_reference("boltz2", "input", "run_boltz2_affinity")

# Reuse the MfnG protein + L-tyrosine complex from above (single ligand → binder auto-detected).
affinity_inputs = Boltz2AffinityInput(complexes=[complex])
affinity_config = Boltz2AffinityConfig(
    verbose=False,
    device="cuda",  # Change to "cpu" if no GPU available
    use_msa=True,
    diffusion_samples=1,
    sampling_steps_affinity=200,
    diffusion_samples_affinity=5,
)

affinity_result = run_boltz2_affinity(affinity_inputs, affinity_config)
affinity_structure = affinity_result.structures[0]
print(f"  affinity_pred_value:         {affinity_structure.metrics.affinity_pred_value:.3f}  (log10 IC50 μM; lower = stronger)")
print(f"  affinity_probability_binary: {affinity_structure.metrics.affinity_probability_binary:.3f}  (binder probability)")

**Input** — `Boltz2AffinityInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `complexes` | `list[proto_tools.entities.complex.Complex]` | required | List of complexes to predict structure for containing chains and entity types. |
| `msas` | `list[proto_tools.tools.structure_prediction.shared_data_models.ComplexMSAs] \| None` | `None` | Per-complex MSAs; a bare dict[int, MSA] is coerced to an unpaired ComplexMSAs. |
| `binder_chain` | `proto_tools.entities.structures.selection.SingleChainSelection \| None` | `None` | Ligand chain to score for affinity; None auto-detects the single ligand in each complex. |

Running run_colabfold_search [00:00]

Scoring affinity (Boltz-2):   0%|          | 0/1 [00:00<?, ?complex/s]

  affinity_pred_value:         2.389  (log10 IC50 μM; lower = stronger)
  affinity_probability_binary: 0.386  (binder probability)
